In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

print("=" * 80)
print("SILVER LAYER - DATA CLEANING & TRANSFORMATION")
print("=" * 80)

# ============ READ BRONZE TABLES FROM DATABASE ============
print("\nReading Bronze tables from database...")
df_bronze_csv = spark.table("dm2_credit_risk.bronze_loan_apps")
df_bronze_json = spark.table("dm2_credit_risk.bronze_world_bank")

print(f"bronze_loan_apps loaded: {df_bronze_csv.count():,} rows")
print(f"bronze_world_bank loaded: {df_bronze_json.count():,} rows")

# ============ CLEAN LOAN APPLICATIONS ============
print("\n...CLEANING LOAN APPLICATIONS...")
print("   - Removing null values")
print("   - Removing invalid records")
print("   - Standardizing columns")
print("   - Creating derived metrics")

df_silver_csv = (df_bronze_csv
    .filter(
        (F.col("SK_ID_CURR").isNotNull()) & 
        (F.col("TARGET").isNotNull()) &
        (F.col("AMT_CREDIT").cast("double") > 0) &
        (F.col("AMT_INCOME_TOTAL").cast("double") > 0)
    )
    .select(
        F.col("SK_ID_CURR"),
        F.col("TARGET"),
        F.coalesce(F.col("AMT_CREDIT").cast("double"), F.lit(0.0)).alias("AMT_CREDIT"),
        F.coalesce(F.col("AMT_INCOME_TOTAL").cast("double"), F.lit(0.0)).alias("AMT_INCOME_TOTAL"),
        F.coalesce(F.col("AMT_ANNUITY").cast("double"), F.lit(0.0)).alias("AMT_ANNUITY"),
        F.coalesce(F.col("DAYS_BIRTH").cast("integer"), F.lit(0)).alias("DAYS_BIRTH"),
        F.coalesce(F.col("DAYS_EMPLOYED").cast("integer"), F.lit(0)).alias("DAYS_EMPLOYED"),
        F.coalesce(F.col("CNT_CHILDREN").cast("integer"), F.lit(0)).alias("CNT_CHILDREN"),
        F.coalesce(F.col("NAME_FAMILY_STATUS"), F.lit("Unknown")).alias("family_status"),
        F.coalesce(F.col("NAME_EDUCATION_TYPE"), F.lit("Unknown")).alias("education_type"),
        F.coalesce(F.col("OCCUPATION_TYPE"), F.lit("Unknown")).alias("occupation_type"),
        F.coalesce(F.col("NAME_CONTRACT_TYPE"), F.lit("Unknown")).alias("contract_type"),
        F.when(F.col("AMT_INCOME_TOTAL").cast("double") > 0, 
               F.round(F.col("AMT_CREDIT").cast("double") / F.col("AMT_INCOME_TOTAL").cast("double"), 2)
        ).otherwise(0).alias("credit_to_income_ratio"),
        F.current_timestamp().alias("processing_date")
    )
)

# Save Silver CSV table
df_silver_csv.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dm2_credit_risk.silver_loan_apps")
print(f"\n silver_loan_apps created: {df_silver_csv.count():,} rows")

# ============ CLEAN WORLD BANK DATA ============
print("\n  CLEANING WORLD BANK DATA...")
print("   - Removing null values")
print("   - Standardizing country codes")
print("   - Converting data types")
print("   - Filtering recent years")

df_silver_json = (df_bronze_json
    .filter(
        (F.col("gdp").isNotNull()) & 
        (F.col("year").isNotNull()) &
        (F.col("country").isNotNull())
    )
    .select(
        F.upper(F.col("country")).alias("country_code"),
        F.col("country_name"),
        F.col("year").cast(IntegerType()).alias("year"),
        F.round(F.col("gdp").cast(DoubleType()), 2).alias("gdp_usd"),
        F.current_timestamp().alias("processing_date")
    )
    .filter(F.col("year") >= 2018)
)

# Save Silver JSON table
df_silver_json.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dm2_credit_risk.silver_world_bank")
print(f"\n silver_world_bank created: {df_silver_json.count():,} rows")

# ============ DISPLAY RESULTS ============
print("\n" + "=" * 80)
print(" SILVER LAYER COMPLETE")
print("=" * 80)

print("\n Loan Applications Sample:")
df_silver_csv.show(5, truncate=False)

print("\n World Bank Data Sample:")
df_silver_json.show(5, truncate=False)

print("\n Ready for GOLD LAYER!")